# logit-thermo A2-4：結晶を止めると T_0 は動くか — データ集め（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。
- **ランタイム：L4**（ハイメモリはいりません）。見込みは **3〜5 時間**（3 モデル × 3 つまみ × 15 温度 × 2 本 × 40 記事 × 512 トークン）。
- **ゲート `docs/A2_4_GATE.md` v0.2（凍結 2026-10-28 18:28）**。数字を集めて保存するだけで、何も判定しません。結果も表示しません。
- 順番：①自己テスト（A2-2 のものと、つまみの確かめ）②BBC の 40 本（A2-2 と同じことを確かめる）③モデルごとに「つまみなし → 写しの禁止 → 偽薬」（Pythia-1B → OLMo-2-1B → Qwen2.5-1.5B）。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）と `scripts/a2_4_collect.py`（`6ef87866ae7d895d…`）をそのまま埋め込み。TF32 は切ります。
- 結果の置き場所：`MyDrive/logit-thermo/A2_4/`。途中で切れても「すべてのセルを実行」で、終わったものは飛ばして続きから回ります。
- **通知**：ntfy の `YOUR_NTFY_TOPIC` に 🚀・回ごとの ✅・🎉・⚠️（エラーの文面まるごと）。名札は「logit-thermo A2-4 データ集め（写しの禁止と偽薬、3 モデル × 3、L4）」。
- **終わったら自動でランタイムを切ります**。エラーで止まったときも、通知してから切ります（止めたいときは最初に `AUTO_DISCONNECT = False`）。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-4 データ集め（写しの禁止と偽薬、3 モデル × 3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-4 collect', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU が足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
print('GPU：', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード（a2_2_collect.py と a2_4_collect.py をそのまま。同じ名前空間で実行）
import hashlib
SRC22 = '"""A2-2（ゲート v0.3）のデータ集め。GPU のノートブックにこのファイルの中身をそのまま埋め込む。\n\n何も判定しない（分析は別の CPU ノートブック）。各モデルについて：\n- 人の文脈（teacher forcing）：各位置・各温度の ℓ（裾の取り分の log-odds）、1 位の log-odds、エントロピー。\n- 温度計の材料：β ＝ 1/T の格子ごとに Σ(⟨z⟩_β − z_y)（人のトークンの平均エネルギーとの差）。\n- a_loc・h（T ＝ 1、人の文脈の各位置）。\n- 生成：各温度・各サンプルで 512 歩、各一歩の ℓ・1 位の log-odds・エントロピーと、選んだトークン。\nすべて float64 で計算し、float32 で保存する。使わない番号（tokenizer の長さ以上）と EOS は −∞ にして外す。\n"""\nimport gc, hashlib, json, math, os, re, time\nimport numpy as np\nimport torch\n\nH = 100\nPROMPT, CONT, N_DOCS, SAMPLES = 64, 512, 40, 2\nTEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\nTCAL_T = np.exp(np.linspace(np.log(0.4), np.log(2.5), 61))   # 温度計の格子（T）\nSEED = 20261027\nMODELS = [\n    (\'pythia-1b\', \'EleutherAI/pythia-1b\', \'Pythia\'),\n    (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\', \'Qwen\'),\n    (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\', \'Qwen\'),\n    (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\', \'Qwen\'),\n    (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\', \'OLMo\'),\n    (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\', \'SmolLM\'),\n    (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\', \'SmolLM\'),\n]\nCROSS = [(\'pythia-1b\', \'olmo2-1b\'), (\'qwen2.5-1.5b\', \'smollm2-1.7b\')]\nBBC_MONTHS = [\'2025-04\', \'2025-05\', \'2025-06\']\nBOILER = re.compile(r\'(Getty Images|Image source|Image caption|Related topics|Follow BBC|^Watch:|^Listen:|To play this video you need to enable JavaScript)\')\nH_FRAC = 0.00066   # H を語彙の割合にした版（ゲート §2.1）\n\n\ndef h2_of(n_vocab):\n    return max(2, int(round(H_FRAC * n_vocab)))\n\n\n# ---------- 一歩の量（torch、float64） ----------\n\ndef step_stats(z, T, valid, H2):\n    """z：(B, V) の logit、valid：(V,) の bool。戻り値：ℓ（順位 H 以降）、1 位の log-odds、エントロピー、ℓ（順位 H2 以降）。\n    どれも (B,)、float64。1 位の log-odds は「1 位を除いた logsumexp」から直接計算する（p1 → 1 でも有限）。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\')) / T\n    lse = torch.logsumexp(z, dim=-1)\n    top = torch.topk(z, max(H, H2) - 1, dim=-1)\n\n    def ell_of(h):\n        head = torch.logsumexp(top.values[:, :h - 1], dim=-1)\n        return torch.logsumexp(z.scatter(-1, top.indices[:, :h - 1], float(\'-inf\')), dim=-1) - head\n    rest = torch.logsumexp(z.scatter(-1, top.indices[:, :1], float(\'-inf\')), dim=-1)\n    top1 = top.values[:, 0] - rest\n    lp = z - lse[:, None]\n    p = torch.exp(lp)\n    ent = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)\n    return ell_of(H), top1, ent, ell_of(H2)\n\n\ndef np_step_stats(z, T, valid, H2):\n    """自己テスト用の素直な numpy 版（1 行）。"""\n    z = np.where(valid, z.astype(np.float64), -np.inf) / T\n    zs = np.sort(z[np.isfinite(z)])[::-1]\n    def lse(x):\n        m = x.max(); return m + np.log(np.exp(x - m).sum())\n    ell = lse(zs[H - 1:]) - lse(zs[:H - 1])\n    ell2 = lse(zs[H2 - 1:]) - lse(zs[:H2 - 1])\n    lp = zs - lse(zs)\n    return ell, float(zs[0] - lse(zs[1:])), float(-(np.exp(lp) * lp).sum()), ell2\n\n\ndef shape_stats(z, valid):\n    """T ＝ 1 の a_loc（順位 300〜3,000 の傾き）と h（頭の確率 ÷ 裾の確率の log10）。z：(N, V)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zs = torch.sort(z, dim=-1, descending=True).values\n    r = torch.arange(300, 3001, device=z.device, dtype=torch.float64)\n    x = torch.log(r); x = x - x.mean()\n    y = zs[:, 299:3000]\n    a = -((y - y.mean(-1, keepdim=True)) * x).sum(-1) / (x * x).sum()\n    h = (torch.logsumexp(zs[:, :H - 1], -1) - torch.logsumexp(zs[:, H - 1:], -1)) / math.log(10)\n    return a, h\n\n\ndef tcal_terms(z, y, valid):\n    """温度計の材料：各 T（TCAL_T）で Σ_t(⟨z⟩_T − z_y)。z：(N, V)、y：(N,)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zy = z.gather(-1, y[:, None])[:, 0]\n    out = []\n    for T in TCAL_T:\n        lp = torch.log_softmax(z / T, dim=-1)\n        p = torch.exp(lp)\n        mean_z = torch.where(p > 0, p * z, torch.zeros_like(p)).sum(-1)\n        out.append(float((mean_z - zy).sum()))\n    return np.array(out)\n\n\n# ---------- 書き出しの文章 ----------\n\ndef clean(text):\n    paras = [p.strip() for p in text.split(\'\\n\')]\n    keep = [p for p in paras if len(p) >= 60 and not BOILER.search(p)]\n    return \'\\n\\n\'.join(keep)\n\n\ndef load_docs():\n    from datasets import load_dataset\n    pool = []\n    for m in BBC_MONTHS:\n        ds = load_dataset(\'RealTimeData/bbc_news_alltime\', m, split=\'train\')\n        for row in ds:\n            c = clean(row[\'content\'] or \'\')\n            if len(c) >= 4000:\n                pool.append({\'month\': m, \'link\': row[\'link\'], \'text\': c})\n    pool.sort(key=lambda d: hashlib.sha256(d[\'link\'].encode()).hexdigest())\n    rng = np.random.default_rng(SEED)\n    idx = rng.choice(len(pool), size=N_DOCS, replace=False)\n    docs = [pool[i] for i in sorted(idx)]\n    meta = {\'pool_size\': len(pool), \'docs\': [{\'month\': d[\'month\'], \'link_sha256\': hashlib.sha256(d[\'link\'].encode()).hexdigest(),\n                                               \'text_sha256\': hashlib.sha256(d[\'text\'].encode()).hexdigest(),\n                                               \'chars\': len(d[\'text\'])} for d in docs]}\n    return docs, meta\n\n\n# ---------- モデルごと ----------\n\ndef load_model(hf_id, device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    return tok, model\n\n\ndef valid_mask(tok, model, device):\n    V = model.get_output_embeddings().weight.shape[0]\n    valid = torch.zeros(V, dtype=torch.bool, device=device)\n    valid[:len(tok)] = True\n    eos = set()\n    if tok.eos_token_id is not None:\n        eos.add(int(tok.eos_token_id))\n    ge = getattr(model, \'generation_config\', None)\n    if ge is not None and ge.eos_token_id is not None:\n        eos |= set([ge.eos_token_id] if isinstance(ge.eos_token_id, int) else ge.eos_token_id)\n    eot = tok.convert_tokens_to_ids(\'<|endoftext|>\') if \'<|endoftext|>\' in tok.get_vocab() else None\n    if eot is not None:\n        eos.add(int(eot))   # 文書の終わりの記号も外す（SmolLM2-Instruct は EOS が <|im_end|> なので、ここで足す）\n    for e in eos:\n        if e < V:\n            valid[e] = False\n    return valid, sorted(eos)\n\n\ndef split_doc(tok, text):\n    ids = tok(text, add_special_tokens=False).input_ids\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    if len(ids) < PROMPT + CONT:\n        return None\n    return bos + ids[:PROMPT], ids[PROMPT:PROMPT + CONT]\n\n\n@torch.no_grad()\ndef teacher_forced(model, prompt, human, valid, device, H2):\n    x = torch.tensor([prompt + human], device=device)\n    logits = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].float()\n    y = torch.tensor(human, device=device)\n    res = {}\n    for k, T in enumerate(TEMPS):\n        res[k] = [v.float().cpu().numpy() for v in step_stats(logits, T, valid, H2)]\n    a, h = shape_stats(logits, valid)\n    return res, a.float().cpu().numpy(), h.float().cpu().numpy(), tcal_terms(logits, y, valid)\n\n\n@torch.no_grad()\ndef generate(model, prompts, T, valid, gen, H2):\n    """prompts：同じ長さのトークン列のリスト。512 歩を KV キャッシュで書く。"""\n    x = torch.tensor(prompts, device=valid.device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    L, T1, E, L2, toks = [], [], [], [], []\n    for t in range(CONT):\n        ell, top1, ent, ell2 = step_stats(logits, T, valid, H2)\n        L.append(ell); T1.append(top1); E.append(ent); L2.append(ell2)\n        z = logits.double().masked_fill(~valid, float(\'-inf\')) / T\n        p = torch.softmax(z, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        toks.append(nxt[:, 0])\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda xs: torch.stack(xs, 1).float().cpu().numpy()\n    return st(L), st(T1), st(E), st(L2), torch.stack(toks, 1).int().cpu().numpy()\n\n\ndef run_model(name, hf_id, docs, outdir, device, notify=print):\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nT, nD = len(TEMPS), len(use)\n    tf = np.zeros((4, nD, nT, CONT), np.float32)   # [ℓ(H), 1 位の log-odds, エントロピー, ℓ(H2)]\n    a_all = np.zeros((nD, CONT), np.float32); h_all = np.zeros((nD, CONT), np.float32)\n    tcal = np.zeros((nD, len(TCAL_T)))\n    for j, i in enumerate(use):\n        res, a, h, tc = teacher_forced(model, *pairs[i], valid, device, H2)\n        for k in range(nT):\n            for q in range(4):\n                tf[q, j, k] = res[k][q]\n        a_all[j], h_all[j], tcal[j] = a, h, tc\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    gen = np.zeros((4, nT, SAMPLES, nD, CONT), np.float32)\n    toks = np.zeros((nT, SAMPLES, nD, CONT), np.int32)\n    for k, T in enumerate(TEMPS):\n        for s in range(SAMPLES):\n            g = torch.Generator(device=device).manual_seed(SEED + 1000 * k + s)\n            l, t1, e, l2, tk = generate(model, prompts, T, valid, g, H2)\n            gen[0, k, s], gen[1, k, s], gen[2, k, s], gen[3, k, s], toks[k, s] = l, t1, e, l2, tk\n        print(f\'  {name} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    np.savez_compressed(path + \'.part.npz\', tf=tf, a=a_all, h=h_all, tcal=tcal, tcal_T=TCAL_T, gen=gen, toks=toks,\n                        temps=np.array(TEMPS), use=np.array(use), eos=np.array(eos), tok_len=len(tok),\n                        prompt_len=list(plen)[0], revision=str(getattr(model.config, \'_commit_hash\', \'\')),\n                        seconds=time.time() - t0, H2=H2)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n\n\n@torch.no_grad()\ndef cross_model(reader, writer, docs, outdir, device):\n    """reader が writer の書いた文を読む。文字の窓で、人の続きを読んだときと比べる（記録用）。"""\n    path = f\'{outdir}/cross_{reader}_reads_{writer}.npz\'\n    if os.path.exists(path):\n        return\n    from transformers import AutoTokenizer\n    hf = dict((n, h) for n, h, _ in MODELS)\n    wtok = AutoTokenizer.from_pretrained(hf[writer])\n    W = np.load(f\'{outdir}/{writer}.npz\')\n    tok, model = load_model(hf[reader], device)\n    valid, _ = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    use = list(W[\'use\']); Wt = W[\'toks\']\n    out = np.full((2, len(TEMPS), SAMPLES, len(use)), np.nan)   # [cross, human] の ℓ の平均\n    for j, i in enumerate(use):\n        text = docs[i][\'text\']\n        enc = wtok(text, add_special_tokens=False, return_offsets_mapping=True)\n        pend = enc[\'offset_mapping\'][PROMPT - 1][1]                  # 書き手の書き出しの終わりの文字位置\n        pre = text[:pend]\n        for k, T in enumerate(TEMPS):\n            for s in range(SAMPLES):\n                gtxt = wtok.decode(Wt[k, s, j])\n                C = min(len(gtxt), len(text) - pend)\n                for q, cont in enumerate([gtxt[:C], text[pend:pend + C]]):\n                    e2 = tok(pre + cont, add_special_tokens=False, return_offsets_mapping=True)\n                    ids = e2[\'input_ids\']; offs = e2[\'offset_mapping\']\n                    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n                    x = torch.tensor([bos + ids], device=device)\n                    logits = model(x).logits[0].float()\n                    pos = [n for n, (a0, _) in enumerate(offs) if pend <= a0 < pend + C and n > 0]\n                    if not pos:\n                        continue\n                    pred = torch.tensor([n - 1 + len(bos) for n in pos], device=device)\n                    ell = step_stats(logits[pred], T, valid, H2)[0]\n                    out[q, k, s, j] = float(ell.mean())\n    np.savez_compressed(path + \'.part.npz\', out=out, temps=np.array(TEMPS))\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n# ---------- 自己テスト（本番の前に） ----------\n\n@torch.no_grad()\ndef self_test(device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    H2 = 33\n    # ① 一歩の量が numpy の素直な計算と一致する（1 位が飛び抜けた行も入れる：1 位の log-odds が有限か）\n    z = torch.randn(3, valid.shape[0], device=device) * 3\n    z[2, 7] += 80.0\n    for T in [0.6, 1.0, 1.5]:\n        got_all = step_stats(z, T, valid, H2)\n        assert all(bool(torch.isfinite(v).all()) for v in got_all), (\'有限でない値\', T)\n        for b in range(3):\n            ref = np_step_stats(z[b].cpu().numpy(), T, valid.cpu().numpy(), H2)\n            got = tuple(float(v[b]) for v in got_all)\n            assert all(abs(g - r) < 1e-6 * max(1, abs(r)) for g, r in zip(got, ref)), (T, got, ref)\n    # ② KV キャッシュの生成と、まとめて計算した logit が一致する\n    g = torch.Generator(device=device).manual_seed(0)\n    prompts = [list(range(5, 5 + 16)), list(range(30, 30 + 16))]\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    seq, steps = x, []\n    for t in range(8):\n        nxt = torch.multinomial(torch.softmax(lg.masked_fill(~valid, float(\'-inf\')), -1), 1, generator=g)\n        steps.append(lg.clone()); seq = torch.cat([seq, nxt], 1)\n        out = model(nxt, past_key_values=past, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    full = model(seq).logits\n    for t in range(8):\n        assert torch.allclose(full[:, 15 + t, :], steps[t], atol=1e-4), (\'KV キャッシュの不一致\', t)\n    # ③ 1 歩目は生成と人の文脈で同じ分布\n    human = list(range(100, 100 + 64))   # 小さなモデルは位置 512 までなので短くする（使うのは位置 15 だけ）\n    tf_logits = model(torch.tensor([prompts[0] + human], device=device)).logits[0, 15]\n    gen_first = model(torch.tensor([prompts[0]], device=device)).logits[0, -1]\n    assert torch.allclose(tf_logits, gen_first, atol=1e-5), \'1 歩目が一致しない\'\n    return \'OK\'\n'
SRC24 = '"""A2-4（ゲート v0.2、凍結 2026-10-28 18:28）のデータ集め。GPU（L4）のノートブックで、scripts/a2_2_collect.py を\n実行したのと同じ名前空間でこのファイルを実行する（A2-2 の関数をそのまま使う）。\n\nつまみ（ゲート §2）：\n- \'none\'：つまみなし（λ ＝ 0）。\n- \'block\'：写しの禁止（λ ＝ ∞）。直前の 3 トークン (a, b, c) がそれまでの文（書き出しを含む）に出ていたら、\n  そのとき続いたトークン d を禁止する（logit を −∞）。no_repeat_ngram_size ＝ 4 と同じ。\n- \'placebo\'：偽薬。写しかけ歩（禁止の候補がある歩）でない歩から歩を選び、その歩のつまみ前の 1 位を禁止する。\n  全体の禁止の割合が p(T) になるように、各歩で「ここまでに禁止しておきたい数の残り」（owed：毎歩 p(T) × 行の数を足し、\n  実際に禁止した数を引く）を、写しかけ歩でない行それぞれに確率 min(1, owed ÷ 写しかけ歩でない行の数) で割り当てる\n  （足りなかった分は次の歩に持ち越す。データを開く前に決めた細部。ゲート §6 の 4 の確かめはこの割合で行う）。\n  p(T) は \'block\' の生成で「禁止したトークンがつまみ前の 1 位だった歩」の割合（温度ごと、全記事・全サンプル・全歩）。\n  歩の選び方の乱数は、生成の乱数と別の種（numpy、PLACEBO_SEED）で決める。\nつまみは温度で割る前の logit にかける。\n\n物差し（ゲート §3）：ℓ・1 位の log-odds・エントロピー・ℓ(H2) は、つまみをかける前の logit から計算する（本命）。\n記録として、つまみをかけたあとの logit から ℓ（H ＝ 100）も計算する。\n歩ごとの印：写しかけ歩か（cand）、禁止の候補のつまみ前の確率の合計（cand_mass）、実際に禁止したか（banned）、\n禁止したトークンにつまみ前の 1 位が入っていたか（banned_top1）、実際に禁止した確率の合計（ban_mass、つまみ前）。\n確率はその温度の分布（logit ÷ T の softmax、外した番号を除く）で測る。\n人の文脈は A2-2 と同じ（つまみはかからない）。人の文のトークン（書き出しと続き）も保存する（分析で写しかけの印を作るため）。\n判定はしない（分析は scripts/a2_4_analyze.py）。\n"""\nimport gc, json, math, os, time\nimport numpy as np\nimport torch\n\nSEED4 = 20261029\nPLACEBO_SEED = 20261030\nMODES = [\'none\', \'block\', \'placebo\']\nCONDITIONS = [  # 名前, Hugging Face の ID\n    (\'pythia-1b\', \'EleutherAI/pythia-1b\'),\n    (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'),\n    (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'),\n]\n\n\nclass CopyIndex:\n    """1 本の文の「3 トークン → それまでに続いたトークンの集合」。"""\n\n    def __init__(self, hist):\n        self.seq = [int(t) for t in hist]\n        self.nxt = {}\n        for i in range(3, len(self.seq)):\n            self.nxt.setdefault(tuple(self.seq[i - 3:i]), set()).add(self.seq[i])\n\n    def candidates(self):\n        if len(self.seq) < 3:\n            return ()\n        return tuple(self.nxt.get(tuple(self.seq[-3:]), ()))\n\n    def push(self, t):\n        t = int(t)\n        if len(self.seq) >= 3:\n            self.nxt.setdefault(tuple(self.seq[-3:]), set()).add(t)\n        self.seq.append(t)\n\n\n@torch.no_grad()\ndef generate_pen(model, prompts, T, valid, gen, H2, mode, p_placebo=0.0, rng=None):\n    """prompts：同じ長さのトークン列のリスト。512 歩を KV キャッシュで書く。戻り値は dict（numpy）。"""\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndex(p) for p in prompts]\n    owed = 0.0\n    rec = {k: [] for k in [\'ell\', \'top1\', \'ent\', \'ell2\', \'ell_pen\', \'cand_mass\', \'ban_mass\', \'cand\', \'banned\', \'banned_top1\', \'toks\']}\n    for t in range(CONT):\n        ell, top1, ent, ell2 = step_stats(logits, T, valid, H2)          # つまみ前の logit（本命）\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        lp = torch.log_softmax(z / T, dim=-1)                             # その温度の分布（つまみ前）\n        argmax = z.argmax(-1)\n        cands = [idx[b].candidates() for b in range(B)]\n        cand = torch.tensor([len(c) > 0 for c in cands], device=device)\n        cm = torch.zeros(B, dtype=torch.float64, device=device)\n        rows, cols = [], []\n        for b, c in enumerate(cands):\n            if c:\n                cc = torch.tensor(c, device=device)\n                cm[b] = torch.exp(torch.logsumexp(lp[b, cc], 0))\n                if mode == \'block\':\n                    rows += [b] * len(c); cols += list(c)\n        if mode == \'placebo\':\n            nc = ~cand.cpu().numpy()\n            owed += p_placebo * B                                         # この歩までに禁止しておきたい数の残り\n            q = min(1.0, owed / nc.sum()) if nc.any() else 0.0\n            pick = (rng.random(B) < q) & nc\n            owed -= int(pick.sum())\n            for b in np.nonzero(pick)[0]:\n                rows.append(int(b)); cols.append(int(argmax[b]))\n        banned = torch.zeros(B, dtype=torch.bool, device=device)\n        banned_top1 = torch.zeros(B, dtype=torch.bool, device=device)\n        ban_mass = torch.zeros(B, dtype=torch.float64, device=device)\n        zp = z.clone()\n        if rows:\n            r = torch.tensor(rows, device=device); c = torch.tensor(cols, device=device)\n            zp[r, c] = float(\'-inf\')\n            banned[r] = True\n            banned_top1[r[c == argmax[r]]] = True\n            ban_mass.index_add_(0, r, torch.exp(lp[r, c]))\n        ell_pen = step_stats(zp.float(), T, valid, H2)[0]                 # 記録：つまみ後の logit の ℓ\n        p = torch.softmax(zp / T, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        for k, v in [(\'ell\', ell), (\'top1\', top1), (\'ent\', ent), (\'ell2\', ell2), (\'ell_pen\', ell_pen), (\'cand_mass\', cm),\n                     (\'ban_mass\', ban_mass), (\'cand\', cand), (\'banned\', banned), (\'banned_top1\', banned_top1), (\'toks\', nxt[:, 0])]:\n            rec[k].append(v)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    res = {}\n    for k, v in rec.items():\n        a = torch.stack(v, 1)\n        res[k] = a.cpu().numpy().astype(np.int32 if k == \'toks\' else (np.bool_ if a.dtype == torch.bool else np.float32))\n    return res\n\n\ndef run_mode(name, hf_id, docs, outdir, device, mode, notify=print):\n    path = f\'{outdir}/{name}@{mode}.npz\'\n    if os.path.exists(path):\n        print(name, mode, \'はもう終わっています（飛ばします）\')\n        return\n    p_pl = None\n    if mode == \'placebo\':\n        B = np.load(f\'{outdir}/{name}@block.npz\')\n        p_pl = B[\'g_banned_top1\'].reshape(B[\'g_banned_top1\'].shape[0], -1).mean(1)     # (nT,)\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nT, nD = len(TEMPS), len(use)\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    humans = [pairs[i][1] for i in use]\n    tf = np.zeros((4, nD, nT, CONT), np.float32)\n    if mode == \'none\':   # 人の文脈（A2-2 と同じ。つまみはかからない）は \'none\' でだけ計算する\n        for j, i in enumerate(use):\n            res, a, h, tc = teacher_forced(model, *pairs[i], valid, device, H2)\n            for k in range(nT):\n                for q in range(4):\n                    tf[q, j, k] = res[k][q]\n    keys_f = [\'ell\', \'top1\', \'ent\', \'ell2\', \'ell_pen\', \'cand_mass\', \'ban_mass\']\n    keys_b = [\'cand\', \'banned\', \'banned_top1\']\n    G = {k: np.zeros((nT, SAMPLES, nD, CONT), np.float32) for k in keys_f}\n    G.update({k: np.zeros((nT, SAMPLES, nD, CONT), np.bool_) for k in keys_b})\n    toks = np.zeros((nT, SAMPLES, nD, CONT), np.int32)\n    for k, T in enumerate(TEMPS):\n        for s in range(SAMPLES):\n            g = torch.Generator(device=device).manual_seed(SEED4 + 1000 * k + s)\n            rng = np.random.default_rng(PLACEBO_SEED + 1000 * k + s)\n            r = generate_pen(model, prompts, T, valid, g, H2, mode, 0.0 if p_pl is None else float(p_pl[k]), rng)\n            for kk in keys_f + keys_b:\n                G[kk][k, s] = r[kk]\n            toks[k, s] = r[\'toks\']\n        print(f\'  {name}@{mode} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    extra = {} if p_pl is None else {\'p_placebo\': p_pl}\n    np.savez_compressed(path + \'.part.npz\', tf=tf, toks=toks, temps=np.array(TEMPS), use=np.array(use), eos=np.array(eos),\n                        tok_len=len(tok), prompt_len=list(plen)[0], prompts=np.array(prompts, np.int32),\n                        humans=np.array(humans, np.int32), revision=str(getattr(model.config, \'_commit_hash\', \'\')),\n                        seconds=time.time() - t0, H2=H2, mode=mode, **{f\'g_{k}\': v for k, v in G.items()}, **extra)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name}@{mode} おわり（{time.time() - t0:.0f} 秒）\')\n\n\n@torch.no_grad()\ndef self_test_pen(device):\n    """小さなランダムのモデルで：つまみ前の logit を記録しているか、禁止が効いているか、偽薬が写しかけ歩を避けるか。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    H2 = 33\n    global CONT\n    keep = CONT\n    CONT = 40\n    try:\n        # くり返しの多い書き出し（写しかけ歩が出やすい）\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4]\n        for T in [0.5, 1.0]:\n            g = torch.Generator(device=device).manual_seed(0)\n            r = generate_pen(model, prompts, T, valid, g, H2, \'block\')\n            # ① 写し歩が 0（4 トークンの並びがそれまでに出ていない）\n            for b in range(2):\n                seq = list(prompts[b]) + list(r[\'toks\'][b])\n                seen = set()\n                for i in range(3, len(seq)):\n                    gram = tuple(seq[i - 3:i + 1])\n                    if i >= len(prompts[b]):\n                        assert gram not in seen, (\'写しが残っている\', T, b, i)\n                    seen.add(gram)\n            assert r[\'cand\'].any(), \'写しかけ歩が 1 つもない（確かめにならない）\'\n            # ② 記録した ℓ は、つまみ前の logit と一致する（同じバッチの形で KV キャッシュで流し直し、相対 1e-6。ゲート §6 の 2）。\n            #    まとめて計算し直した値とも近い（絶対 1e-3）。禁止した歩ではつまみ後と違う\n            xx = torch.tensor(prompts, device=device)\n            o = model(xx, use_cache=True); pk, lg = o.past_key_values, o.logits[:, -1, :].float()\n            for t in range(CONT):\n                e = step_stats(lg, T, valid, H2)[0].cpu().numpy()\n                assert np.allclose(e, r[\'ell\'][:, t], rtol=1e-6, atol=1e-9), (\'流し直した ℓ と一致しない\', T, t)\n                nx = torch.tensor(r[\'toks\'][:, t:t + 1], device=device, dtype=torch.long)\n                o = model(nx, past_key_values=pk, use_cache=True); pk, lg = o.past_key_values, o.logits[:, -1, :].float()\n            for b in range(2):\n                seq = torch.tensor([list(prompts[b]) + list(r[\'toks\'][b])], device=device)\n                full = model(seq).logits[0, len(prompts[b]) - 1: len(prompts[b]) - 1 + CONT].float()\n                ref = step_stats(full, T, valid, H2)[0].cpu().numpy()\n                assert np.allclose(ref, r[\'ell\'][b], atol=1e-3), (\'つまみ前の ℓ と一致しない\', T, b)\n                m = r[\'banned\'][b] & (r[\'ban_mass\'][b] > 1e-3)\n                if m.any():\n                    assert np.all(np.abs(r[\'ell_pen\'][b][m] - r[\'ell\'][b][m]) > 1e-6), (\'禁止した歩でつまみ後の ℓ が同じ\', T, b)\n            # ③ 偽薬：写しかけ歩では禁止しない。p ＝ 1 なら、写しかけ歩でない歩は全部 1 位を禁止する\n            g = torch.Generator(device=device).manual_seed(0)\n            rp = generate_pen(model, prompts, T, valid, g, H2, \'placebo\', 1.0, np.random.default_rng(0))\n            assert not (rp[\'banned\'] & rp[\'cand\']).any(), \'偽薬が写しかけ歩で禁止した\'\n            assert (rp[\'banned\'] == ~rp[\'cand\']).all() and (rp[\'banned_top1\'] == rp[\'banned\']).all(), \'偽薬の禁止が狙いどおりでない\'\n            # ④ つまみなし：何も禁止しない。1 歩目のつまみ前の ℓ は 3 つで同じ\n            g = torch.Generator(device=device).manual_seed(0)\n            rn = generate_pen(model, prompts, T, valid, g, H2, \'none\')\n            assert not rn[\'banned\'].any()\n            assert np.allclose(rn[\'ell\'][:, 0], r[\'ell\'][:, 0]) and np.allclose(rn[\'ell\'][:, 0], rp[\'ell\'][:, 0])\n    finally:\n        CONT = keep\n    return \'OK\'\n\n\n@torch.no_grad()\ndef self_test_run_mode(device, tmpdir=\'/tmp/a2_4_selftest\'):\n    """小さなランダムのモデルで run_mode を 3 つのつまみで通しで回し、分析が読むキーがそろっているか確かめる。"""\n    import shutil\n    global PROMPT, CONT, TEMPS, shape_stats\n    keep = (PROMPT, CONT, TEMPS, shape_stats)\n    PROMPT, CONT, TEMPS = 8, 20, [0.7, 1.0]\n    shape_stats = lambda z, valid: (torch.zeros(z.shape[0]), torch.zeros(z.shape[0]))   # 小さな語彙では順位 300〜3,000 がない\n    shutil.rmtree(tmpdir, ignore_errors=True); os.makedirs(tmpdir)\n    try:\n        rng = np.random.default_rng(0)\n        words = [\'alpha\', \'beta\', \'gamma\', \'delta\', \'eps\', \'zeta\', \'eta\', \'theta\']\n        docs = [{\'text\': \' \'.join(rng.choice(words, 200))} for _ in range(3)]\n        hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n        for mode in MODES:\n            run_mode(\'tiny\', hf_id, docs, tmpdir, device, mode)\n        need = {\'tf\', \'toks\', \'temps\', \'use\', \'prompts\', \'humans\', \'g_ell\', \'g_top1\', \'g_ent\', \'g_ell2\', \'g_ell_pen\',\n                \'g_cand\', \'g_cand_mass\', \'g_banned\', \'g_banned_top1\', \'g_ban_mass\'}\n        for mode in MODES:\n            Z = np.load(f\'{tmpdir}/tiny@{mode}.npz\')\n            miss = need - set(Z.files)\n            assert not miss, (mode, miss)\n            assert Z[\'g_ell\'].shape == (2, SAMPLES, Z[\'use\'].shape[0], 20), Z[\'g_ell\'].shape\n        assert \'p_placebo\' in np.load(f\'{tmpdir}/tiny@placebo.npz\').files\n        assert np.abs(np.load(f\'{tmpdir}/tiny@none.npz\')[\'tf\']).sum() > 0\n    finally:\n        PROMPT, CONT, TEMPS, shape_stats = keep\n        shutil.rmtree(tmpdir, ignore_errors=True)\n    return \'OK\'\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRC24.encode()).hexdigest() == '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57'
exec(SRC22)
exec(SRC24)
print('コードを置きました')


In [ ]:
# セル3：自己テスト（小さなランダムのモデル。ここで止まっても、通知してからランタイムを切る）
try:
    print('A2-2 の自己テスト：', self_test('cuda'))
    print('つまみの自己テスト：', self_test_pen('cuda'))
    print('run_mode の通しの自己テスト：', self_test_run_mode('cuda'))
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 自己テストで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# セル4：記事を選ぶ（A2-2 と同じことを確かめる。中身は表示しない）
import json
OUT = '/content/drive/MyDrive/logit-thermo/A2_4'
os.makedirs(OUT, exist_ok=True)
try:
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 記事を選ぶところで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise
print('BBC：A2-2 と同じ 40 本')


In [ ]:
# セル5：モデルごとに「つまみなし → 写しの禁止 → 偽薬」（終わったものは飛ばす）
import transformers, platform
t_all = time.time()
try:
    for name, hf_id in CONDITIONS:
        for mode in MODES:
            run_mode(name, hf_id, docs, OUT, 'cuda', mode, notify=lambda m: notify('✅ ' + m, tags='white_check_mark'))
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ データ集めで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise
json.dump({'gate': 'docs/A2_4_GATE.md v0.2', 'a2_2_collect_sha256': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'a2_4_collect_sha256': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57',
           'torch': torch.__version__, 'transformers': transformers.__version__, 'python': platform.python_version(),
           'gpu': torch.cuda.get_device_name(0), 'float32_matmul_precision': torch.get_float32_matmul_precision(),
           'seconds_this_session': round(time.time() - t_all, 1)}, open(OUT + '/run_meta.json', 'w'), indent=1)
print('全部おわり')


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：9 回分を Drive に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
